# Task 4: Fourier Neural Operator Classifier

Instead of a CNN or pretrained backbone, this task uses a **Fourier Neural Operator (FNO)** as the feature extractor. FNOs learn in function space by applying convolutions in the Fourier domain via FFT, which lets them capture global structure in a single layer rather than composing many local filters.

This makes them architecturally distinct from all EfficientNet/ResNet approaches and gives a meaningful comparison point: does global spectral learning compete with local spatial hierarchies for lensing substructure classification?

Same dataset and protocol as Task 1 (three-class, same train/val split) so results are directly comparable.

**Task 1 reference (EfficientNet-B3):** Test AUC 0.9778 | Accuracy 89.7%

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as TF
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

_here = Path().resolve()
_root = _here.parent if _here.name == "task4" else _here
sys.path.insert(0, str(_root))

from utils.models import FNOClassifier
from utils.losses import LabelSmoothingCrossEntropy

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device}  |  AMP: {use_amp}")

## How the FNO Works

A standard 2-D convolution computes `(f * g)(x) = sum_y f(y) g(x-y)` locally. The FNO instead applies the convolution theorem: multiply in Fourier space, then transform back. This turns the local filter into a global one that can capture arbitrarily large receptive fields in a single operation.

```
SpectralConv2d:
  x  ->  rfft2  ->  multiply by learned complex weights (first k modes)  ->  irfft2  ->  out
  x  ->  pointwise 1x1 conv (bypass/residual)                            ->  out
                                                                         sum + GELU
```

Only the lowest `modes` Fourier coefficients are kept, which acts as a learned spectral filter. Low modes capture large-scale structure (arcs, rings); higher modes capture fine detail. For lensing images, the arc morphology is the discriminating signal, so a moderate `modes=16` is a reasonable starting point.

Full architecture:
```
Input (1, 150, 150)
  -> Lift: pointwise conv  (1 -> 64 channels)
  -> 4 x FNOBlock          (SpectralConv2d + bypass + GELU)
  -> Project: pointwise conv (64 -> 128 channels)
  -> GlobalAvgPool + Dropout(0.4) + Linear(128 -> 3)
Trainable params: ~8.4M  (trained from scratch, no ImageNet weights)
```

In [ ]:
CLASS_NAMES = ["no", "sphere", "vort"]

class LensingDataset(Dataset):
    def __init__(self, root, split="train", augment=False):
        self.samples, self.augment = [], augment
        split_dir = Path(root) / split
        for label, cls in enumerate(CLASS_NAMES):
            for p in sorted((split_dir / cls).glob("*.npy")):
                self.samples.append((str(p), label))
        counts = np.bincount([l for _, l in self.samples], minlength=3)
        print(f"  {split}: {dict(zip(CLASS_NAMES, counts))}  total={len(self.samples)}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 2: arr = arr[np.newaxis]
        img = torch.from_numpy(arr)
        if self.augment: img = self._augment(img)
        return img, label

    @staticmethod
    def _augment(img):
        if torch.rand(1) < 0.5: img = TF.hflip(img)
        if torch.rand(1) < 0.5: img = TF.vflip(img)
        k = int(torch.randint(0, 4, (1,)))
        if k: img = torch.rot90(img, k, dims=[-2, -1])
        if torch.rand(1) < 0.3:
            c, h, w = img.shape
            sh = int(h * torch.empty(1).uniform_(0.02, 0.15))
            sw = int(w * torch.empty(1).uniform_(0.02, 0.15))
            r0 = int(torch.randint(0, h - sh + 1, (1,)))
            c0 = int(torch.randint(0, w - sw + 1, (1,)))
            img = img.clone()
            img[:, r0:r0+sh, c0:c0+sw] = img.mean()
        return img

def get_sample_weights(ds):
    counts  = np.bincount([l for _, l in ds.samples], minlength=3)
    class_w = 1.0 / (counts.astype(np.float64) + 1e-8)
    return torch.tensor([class_w[l] for _, l in ds.samples])

## Setup

No pretrained weights -- the FNO is trained from scratch on the lensing data. This means we can use a higher learning rate (1e-3 vs 5e-5 for pretrained backbones) and a longer warmup.

In [ ]:
torch.manual_seed(42); np.random.seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

DATA_DIR    = "../task1_data"
EPOCHS      = 60
BATCH_SIZE  = 64
LR          = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN      = 64
MODES       = 16
N_LAYERS    = 4
SMOOTHING   = 0.05
NUM_WORKERS = 4
SAVE_DIR    = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

train_ds = LensingDataset(DATA_DIR, "train", augment=True)
val_ds   = LensingDataset(DATA_DIR, "val",   augment=False)
try:    test_ds = LensingDataset(DATA_DIR, "test",  augment=False)
except: test_ds = val_ds; print("No test split, using val")

in_channels = train_ds[0][0].shape[0]

sampler = WeightedRandomSampler(get_sample_weights(train_ds), len(train_ds.samples), replacement=True)
pin = device.type == "cuda"
kw  = dict(num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=(NUM_WORKERS > 0))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, **kw)

model     = FNOClassifier(in_channels=in_channels, num_classes=3,
                          hidden=HIDDEN, modes=MODES, n_layers=N_LAYERS).to(device)
n_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FNO (hidden={HIDDEN}, modes={MODES}, layers={N_LAYERS}) | params: {n_params:,}")

criterion = LabelSmoothingCrossEntropy(smoothing=SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=EPOCHS,
    pct_start=0.1, anneal_strategy="cos", div_factor=25.0, final_div_factor=1e4,
)
scaler = torch.amp.GradScaler("cuda") if use_amp else None

## Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.amp.autocast("cuda", enabled=(scaler is not None)):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        optimizer.zero_grad()
        if scaler:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device, tta=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    tta_t = [lambda x: x, lambda x: TF.hflip(x), lambda x: TF.vflip(x),
             lambda x: torch.rot90(x, 1, dims=[-2,-1]),
             lambda x: torch.rot90(x, 2, dims=[-2,-1]),
             lambda x: torch.rot90(x, 3, dims=[-2,-1])] if tta else [lambda x: x]
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if tta:
            probs = torch.stack([torch.softmax(model(torch.stack([t(im) for im in imgs])), 1)
                                 for t in tta_t]).mean(0)
            logits = torch.log(probs + 1e-8)
        else:
            logits = model(imgs); probs = torch.softmax(logits, 1)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        correct    += (probs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        all_probs.append(probs.cpu().numpy()); all_labels.append(labels.cpu().numpy())
    return total_loss/total, correct/total, np.concatenate(all_probs), np.concatenate(all_labels)

def compute_roc_auc(probs, labels):
    y_bin = label_binarize(labels, classes=[0,1,2])
    return float(roc_auc_score(y_bin, probs, multi_class="ovr", average="macro"))

def compute_per_class_auc(probs, labels):
    y_bin = label_binarize(labels, classes=[0,1,2])
    return {CLASS_NAMES[i]: float(roc_auc_score(y_bin[:,i], probs[:,i])) for i in range(3)}

## Training Loop

In [ ]:
best_val_auc = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
    scheduler.step()
    val_loss, val_acc, val_probs, val_labels = evaluate(model, val_loader, criterion, device)
    val_auc = compute_roc_auc(val_probs, val_labels)
    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:3d}/{EPOCHS} | Train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f} AUC {val_auc:.4f} | "
          f"LR {lr_now:.2e} | {elapsed:.1f}s")
    history.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
                     "val_loss": val_loss, "val_acc": val_acc, "val_auc": float(val_auc)})
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pt")
        print(f"  New best val AUC: {best_val_auc:.4f}")

model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device))
_, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion, device, tta=True)
test_auc  = compute_roc_auc(test_probs, test_labels)
per_class = compute_per_class_auc(test_probs, test_labels)

print(f"\nTest accuracy: {test_acc:.4f}  |  Test ROC-AUC (macro): {test_auc:.4f}")
print("Per-class AUC:", {k: f"{v:.4f}" for k, v in per_class.items()})
print("\nTask 1 reference: accuracy 0.8972 | AUC 0.9778")

results = {"test_accuracy": test_acc, "test_roc_auc": float(test_auc),
           "per_class_auc": per_class, "best_val_auc": best_val_auc, "history": history}
with open(f"{SAVE_DIR}/results.json", "w") as f: json.dump(results, f, indent=2)

## Training Curves and ROC Comparison

In [ ]:
with open(f"{SAVE_DIR}/results.json") as f: results = json.load(f)
h = results["history"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot([e["epoch"] for e in h], [e["train_loss"] for e in h], label="train")
axes[0].plot([e["epoch"] for e in h], [e["val_loss"]   for e in h], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([e["epoch"] for e in h], [e["train_acc"] for e in h], label="train")
axes[1].plot([e["epoch"] for e in h], [e["val_acc"]   for e in h], label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot([e["epoch"] for e in h], [e["val_auc"] for e in h], color="green")
axes[2].axhline(max(e["val_auc"] for e in h), color="red", linestyle="--", alpha=0.5,
                label=f"best {max(e['val_auc'] for e in h):.4f}")
axes[2].axhline(0.9778, color="gray", linestyle=":", alpha=0.7, label="Task1 EfficientNet 0.9778")
axes[2].set_title("Val ROC-AUC"); axes[2].set_xlabel("Epoch")
axes[2].set_ylim([0.5, 1.0]); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

# Per-class ROC curves
y_bin = label_binarize(test_labels, classes=[0,1,2])
fig, ax = plt.subplots(figsize=(6, 5))
for i, (cls, color) in enumerate(zip(CLASS_NAMES, ["steelblue","tomato","seagreen"])):
    fpr, tpr, _ = roc_curve(y_bin[:,i], test_probs[:,i])
    ax.plot(fpr, tpr, color=color, label=f"{cls} (AUC={results['per_class_auc'][cls]:.4f})")
ax.plot([0,1],[0,1],"k--",alpha=0.4)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"FNO ROC Curves (macro AUC={results['test_roc_auc']:.4f})")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/roc_curves.png", dpi=120, bbox_inches="tight"); plt.show()